# Notebook 3: export the inference vectors as a csv

One row per EPA station. Columns are the features the inference models require, then the two ground
truth columns at the end.

The features come from the grids exported by notebook 1 and notebook 2, read at the cell that
contains each station. No log transform is applied, so `road_density_km_km2` and `n_cars` are raw.
The satellite models use `road_density_log1p` and the pooled model uses `cars_log1p`, both derived
from these columns later.

Ground truth is a caveat worth reading. Both kriging models were trained on the mean between 15:00
and 16:00, taken from one minute sensor series. The EPA files are daily: `temp_f` is a 24
observation daily average and `pm25` is one daily gravimetric sample. So the ground truth here is a
daily mean over the station record, which is not the same construct as the training target. The
column names and the date range columns say so.

## Cell 1: Open the grids from notebook 1 and notebook 2

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

PROJECT_ROOT = Path("/Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis")
NOTEBOOK_FOLDER_NAME = "2026_09_17_create_epa_vectors_for_inference"
PARENT_PATH_DATA = PROJECT_ROOT/"data"/NOTEBOOK_FOLDER_NAME
PARENT_PATH_COMPARISON = PROJECT_ROOT/"data"/"2026_08_26_sensor_comparisons"/"data"

SATELLITE_LAYERS = {
    "ndvi": ("ndvi.geojson", "ndvi"),
    "ndbi": ("ndbi.geojson", "ndbi"),
    "impervious": ("impervious.geojson", "impervious"),
    "road_density_km_km2": ("road_density.geojson", "road_density_km_km2")
}

SVI_LAYERS = {
    "gvi": ("grid_aggregated_gvi.geojson", "gvi_mean"),
    "facade_area": ("grid_aggregated_facade_area.geojson", "facade_area_mean"),
    "n_cars": ("grid_aggregated_n_cars.geojson", "n_cars_mean"),
    "sky_ratio": ("grid_aggregated_sky_ratio.geojson", "sky_ratio_mean"),
    "road_ratio": ("grid_aggregated_road_ratio.geojson", "road_ratio_mean")
}

FEATURE_ORDER = list(SATELLITE_LAYERS) + list(SVI_LAYERS)

layer_grids = {}
for feature_name, (file_name, column_name) in {**SATELLITE_LAYERS, **SVI_LAYERS}.items():
    layer_grid = gpd.read_file(PARENT_PATH_DATA/file_name)
    if column_name not in layer_grid.columns:
        raise SystemExit(f"{file_name} has no column {column_name}: {list(layer_grid.columns)}")
    layer_grids[feature_name] = layer_grid[[column_name, "geometry"]].rename(columns = {column_name: feature_name})
    print(f"{feature_name:20s} {len(layer_grid):5d} cells, non-null {int(layer_grid[column_name].notna().sum()):5d}  <- {file_name}")

GRID_CRS = layer_grids["ndvi"].crs
print(f"\ncrs: {GRID_CRS}")

ndvi                  2806 cells, non-null  2806  <- ndvi.geojson
ndbi                  2806 cells, non-null  2806  <- ndbi.geojson
impervious            2806 cells, non-null  2806  <- impervious.geojson
road_density_km_km2   2806 cells, non-null  2382  <- road_density.geojson
gvi                   2806 cells, non-null   441  <- grid_aggregated_gvi.geojson
facade_area           2806 cells, non-null   441  <- grid_aggregated_facade_area.geojson
n_cars                2806 cells, non-null   441  <- grid_aggregated_n_cars.geojson
sky_ratio             2806 cells, non-null   441  <- grid_aggregated_sky_ratio.geojson
road_ratio            2806 cells, non-null   441  <- grid_aggregated_road_ratio.geojson

crs: EPSG:26986


## Cell 2: Read the ground truth for the two EPA stations

In [2]:
ROXBURY_KEY = "DUDLEY SQUARE ROXBURY"
VON_HILLERN_KEY = "VON HILLERN ST"
EPA_KEYS = [ROXBURY_KEY, VON_HILLERN_KEY]

def convert_from_fahrenheit_to_celsius(value):
    return (value - 32)*(5/9)

temperature_data = pd.read_csv(PARENT_PATH_COMPARISON/"boston_temp_combined.csv")
pm25_data = pd.read_csv(PARENT_PATH_COMPARISON/"boston_pm25_combined.csv")

ground_truth_rows = []
for site_name in EPA_KEYS:
    station_temperature = temperature_data[temperature_data["site_name"] == site_name]
    station_pm25 = pm25_data[pm25_data["site_name"] == site_name]

    ground_truth_rows.append({
        "site_name": site_name,
        "latitude": float(station_pm25["lat"].iloc[0]),
        "longitude": float(station_pm25["lon"].iloc[0]),
        "temperature_c_daily_mean": float(convert_from_fahrenheit_to_celsius(station_temperature["temp_f"]).mean()),
        "pm25_daily_mean": float(station_pm25["pm25"].mean()),
        "n_days_temperature": int(len(station_temperature)),
        "n_days_pm25": int(len(station_pm25)),
        "first_date": min(station_temperature["date"].min(), station_pm25["date"].min()),
        "last_date": max(station_temperature["date"].max(), station_pm25["date"].max())
    })

ground_truth = pd.DataFrame(ground_truth_rows)
print(ground_truth.to_string(index = False))

            site_name  latitude  longitude  temperature_c_daily_mean  pm25_daily_mean  n_days_temperature  n_days_pm25 first_date  last_date
DUDLEY SQUARE ROXBURY 42.329500 -71.082600                 11.400549         5.767023                 790         1787 2024-01-01 2026-03-30
       VON HILLERN ST 42.325186 -71.056061                 11.622839         6.506513                 787         3158 2024-01-01 2026-03-27


## Cell 3: Read each feature at the cell that contains the station and export the csv

In [3]:
stations = gpd.GeoDataFrame(
    ground_truth.copy(),
    geometry = [Point(longitude, latitude) for longitude, latitude in zip(ground_truth["longitude"], ground_truth["latitude"])],
    crs = "EPSG:4326"
).to_crs(GRID_CRS)

for feature_name in FEATURE_ORDER:
    joined = gpd.sjoin(
        stations[["site_name", "geometry"]],
        layer_grids[feature_name],
        how = "left",
        predicate = "within"
    )
    joined = joined[~joined.index.duplicated(keep = "first")]
    stations[feature_name] = joined[feature_name].to_numpy()

GROUND_TRUTH_COLUMNS = ["temperature_c_daily_mean", "pm25_daily_mean"]
LEADING_COLUMNS = ["site_name", "latitude", "longitude", "n_days_temperature", "n_days_pm25", "first_date", "last_date"]

inference_vectors = pd.DataFrame(stations.drop(columns = "geometry"))
inference_vectors = inference_vectors[LEADING_COLUMNS + FEATURE_ORDER + GROUND_TRUTH_COLUMNS]

missing = inference_vectors[FEATURE_ORDER].isna().sum()
if missing.sum() > 0:
    print("features with no value at a station:")
    print(missing[missing > 0].to_string())

PATH_INFERENCE_VECTORS = PARENT_PATH_DATA/"epa_inference_vectors.csv"
inference_vectors.to_csv(PATH_INFERENCE_VECTORS, index = False)

pd.set_option("display.width", 250)
print(inference_vectors.round(4).to_string(index = False))
print(f"\nsaved {PATH_INFERENCE_VECTORS}")

            site_name  latitude  longitude  n_days_temperature  n_days_pm25 first_date  last_date   ndvi    ndbi  impervious  road_density_km_km2    gvi  facade_area  n_cars  sky_ratio  road_ratio  temperature_c_daily_mean  pm25_daily_mean
DUDLEY SQUARE ROXBURY   42.3295   -71.0826                 790         1787 2024-01-01 2026-03-30 0.0772 -0.0215     84.7379              30.3852 0.1325       0.2367 10.5242     0.2297      0.2876                   11.4005           5.7670
       VON HILLERN ST   42.3252   -71.0561                 787         3158 2024-01-01 2026-03-27 0.1162 -0.0169     79.0266              78.3655 0.1367       0.1750  5.6970     0.4294      0.2194                   11.6228           6.5065

saved /Users/legariapena.j/Documents/open-sensing-analysis/open-sensing-analysis/data/2026_09_17_create_epa_vectors_for_inference/epa_inference_vectors.csv
